In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import os

# Define your model class (should be the same as training)
class AnimalCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Dropout2d(0.2),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Dropout2d(0.3),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self._to_linear = None
        self._get_flattened_size()
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self._to_linear, 256), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    
    def _get_flattened_size(self):
        with torch.no_grad():
            x = torch.zeros(1, 3, 128, 128)
            x = self.features(x)
            self._to_linear = x.view(1, -1).shape[1]

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load the saved model and set it to evaluation mode
num_classes = len(dataset.classes)   # Replace with your actual number of classes
model = AnimalCNN(num_classes)
model.load_state_dict(torch.load('animal_model.pth', map_location=device))
model.to(device)
model.eval()

# Class names (replace or load according to your dataset)
class_names = ['cat', 'dog', 'snake']  # Use the actual class names from your dataset

# Image preprocessing (same as training)
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])
])

def predict_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0)  # Add batch dimension
    image = image.to(device)

    with torch.no_grad():
        outputs = model(image)
        _, predicted = torch.max(outputs.data, 1)
        class_idx = predicted.item()
    return class_names[class_idx]

# For single image
image_path = r'C:\Users\ashan\Downloads\dog1.jpg'
predicted_class = predict_image(image_path)
print(f"Predicted class for the single image: {predicted_class}")

# For multiple images
image_folder = r'D:\MY-project\CNN\TestImage'
for img_file in os.listdir(image_folder):
    img_path = os.path.join(image_folder, img_file)
    if img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
        pred = predict_image(img_path)
        print(f"{img_file}: {pred}")

